# LLM Document Analysis: New Product Releases in SEC 8-K Filings

**CAP 5619 – AI in Fintech, Assignment 2**

This notebook pulls SEC Form 8-K filings for S&P 500 companies from EDGAR, uses a large language model
(Anthropic **Claude Sonnet 5.5** through the API) to run Named Entity Recognition on each filing and pull out
newly announced products, and saves the result as a CSV with the columns:

`company_name, stock_name, filing_time, new_product, product_description`

**Pipeline**

1. **Collect**: S&P 500 tickers → SEC CIK numbers → each company's 8-K list (EDGAR Atom feed).
2. **Preprocess**: download each filing's press-release exhibit (EX-99.x), convert HTML → plain text, keep only
   filings whose text mentions a product launch.
3. **Extract (LLM + NER)**: the LLM tags ORGANIZATION and PRODUCT entities and returns the newly announced
   products as structured JSON.
4. **Output**: clean the results (≤180-char descriptions, deduplicate) and write `output/new_products.csv`.

**Setup:** put `ANTHROPIC_API_KEY=...` and `SEC_USER_AGENT="Your Name your@email"` in a `.env` file
(SEC requires a contact in the User-Agent). Install dependencies with `pip install -r requirements.txt`.

In [ ]:
import csv
import json
import os
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from io import StringIO
from pathlib import Path

import anthropic
import pandas as pd
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from pydantic import BaseModel, Field

load_dotenv()  # reads ANTHROPIC_API_KEY and SEC_USER_AGENT from .env

In [ ]:
# ---------------- Configuration ----------------
SEC_HEADERS = {"User-Agent": os.environ["SEC_USER_AGENT"]}  # SEC requires a declared contact
SEC_DELAY = 0.15            # seconds between SEC requests (SEC limit is 10 requests/second)

LLM_MODEL = "claude-sonnet-5-5"   # Anthropic Claude Sonnet 5.5
LLM_EFFORT = "low"                # reasoning effort: extraction is a simple task
LLM_WORKERS = 6                   # parallel LLM requests

FILINGS_PER_COMPANY = 40          # most recent 8-Ks requested per company
MIN_FILING_DATE = "2024-01-01"    # ignore older filings
MAX_CANDIDATE_DOCS = 300          # stop collecting after this many product-related documents
MAX_DOC_CHARS = 20_000            # press releases lead with the announcement; cap length to control cost
MAX_DESC_LEN = 180                # descriptions must be shorter than 180 characters

# 8-K items that can carry product news: 2.02 results, 7.01 Reg FD disclosure, 8.01 other events
RELEVANT_ITEMS = ("2.02", "7.01", "8.01")

# Sectors where product launches are common (banks, utilities, REITs, etc. rarely launch products)
PRODUCT_SECTORS = ["Information Technology", "Health Care", "Communication Services",
                   "Consumer Discretionary", "Consumer Staples", "Industrials"]

CACHE_DIR = Path("cache")         # raw downloads + LLM answers, so re-runs are fast and free
OUTPUT_DIR = Path("output")
for d in (CACHE_DIR / "docs", CACHE_DIR / "llm", OUTPUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

## 1. Collect 8-K filings from SEC EDGAR

* `company_tickers.json` maps each ticker to its CIK (e.g. AAPL → `0000320193`).
* The EDGAR Atom feed `browse-edgar?action=getcompany&CIK=...&type=8-K&output=atom` lists a company's 8-Ks
  with filing date, accession number, item numbers and a link to the filing index. It's parsed with BeautifulSoup.

In [ ]:
def sec_get(url: str) -> requests.Response:
    """GET a SEC URL with the required User-Agent, rate limiting and simple retries."""
    for attempt in range(3):
        time.sleep(SEC_DELAY)
        resp = requests.get(url, headers=SEC_HEADERS, timeout=30)
        if resp.status_code == 200:
            return resp
        time.sleep(2 ** attempt)  # back off on 429/5xx
    resp.raise_for_status()
    return resp


def load_sp500() -> pd.DataFrame:
    """Return the S&P 500 constituents (ticker, name, sector) from Wikipedia."""
    html = requests.get("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
                        headers={"User-Agent": SEC_HEADERS["User-Agent"]}, timeout=30).text
    table = pd.read_html(StringIO(html))[0]
    return table.rename(columns={"Symbol": "ticker", "Security": "name", "GICS Sector": "sector"})[
        ["ticker", "name", "sector"]]


def load_ticker_to_cik() -> dict:
    """Map ticker -> 10-digit zero-padded CIK using SEC's company_tickers.json."""
    data = sec_get("https://www.sec.gov/files/company_tickers.json").json()
    return {row["ticker"]: str(row["cik_str"]).zfill(10) for row in data.values()}


def list_8k_filings(cik: str, count: int = FILINGS_PER_COMPANY) -> list[dict]:
    """Parse the EDGAR Atom feed and return the company's recent 8-K filings."""
    url = (f"https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK={cik}"
           f"&type=8-K&count={count}&output=atom")
    feed = BeautifulSoup(sec_get(url).text, "xml")
    filings = []
    for entry in feed.find_all("entry"):
        content = entry.find("content")
        filings.append({
            "accession": content.find("accession-number").text.strip(),
            "form": content.find("filing-type").text.strip(),
            "filing_date": content.find("filing-date").text.strip(),
            "items": content.find("items-desc").text.strip() if content.find("items-desc") else "",
            "index_url": content.find("filing-href").text.strip(),
        })
    return filings

In [ ]:
sp500 = load_sp500()
ticker_to_cik = load_ticker_to_cik()

# Keep product-oriented sectors; EDGAR uses '-' instead of '.' in tickers (BRK.B -> BRK-B)
companies = sp500[sp500["sector"].isin(PRODUCT_SECTORS)].copy()
companies["cik"] = companies["ticker"].str.replace(".", "-", regex=False).map(ticker_to_cik)
companies = companies.dropna(subset=["cik"]).reset_index(drop=True)
print(f"S&P 500 companies: {len(sp500)} | in product-oriented sectors with a CIK: {len(companies)}")

print(list_8k_filings(companies.loc[0, "cik"], count=10)[:2])  # sanity check on one company

## 2. Preprocess: download press releases and keep product-related filings

The 8-K form itself is mostly boilerplate. The actual announcement sits in the attached press release
(**Exhibit 99.x**), so for each filing we:

1. open the filing index page and find the EX-99 exhibits (falling back to the main 8-K document),
2. convert HTML → plain text with BeautifulSoup,
3. apply a cheap **keyword pre-filter** (launch, introduce, unveil, now available, FDA approval…), so the LLM
   only sees documents that may announce a product. Most 8-Ks cover earnings, executive changes or financing.

In [ ]:
PRODUCT_KEYWORDS = re.compile(
    r"\b(launch(es|ed|ing)?|introduc(es|ed|ing)|unveil(s|ed|ing)?|now available|"
    r"generally available|general availability|FDA approv(al|es|ed)|new (product|platform|model|service)s?)\b",
    re.IGNORECASE)


def html_to_text(html: str) -> str:
    """Strip HTML tags/scripts and collapse whitespace."""
    soup = BeautifulSoup(html, "lxml")
    for tag in soup(["script", "style"]):
        tag.decompose()
    return re.sub(r"\s+", " ", soup.get_text(" ")).strip()


def get_filing_text(index_url: str) -> str:
    """Return the plain text of a filing's EX-99 exhibits (or of the main 8-K document if none)."""
    index = BeautifulSoup(sec_get(index_url).text, "lxml")
    exhibits, main_doc = [], None
    for row in index.select("table.tableFile tr")[1:]:
        cells = row.find_all("td")
        link = row.find("a")
        if len(cells) < 4 or link is None:
            continue
        doc_type = cells[3].get_text(strip=True)
        href = link["href"].replace("/ix?doc=", "")      # inline-XBRL viewer link -> raw document
        if not href.endswith((".htm", ".html", ".txt")):
            continue
        if doc_type.startswith("EX-99"):
            exhibits.append(href)
        elif doc_type.startswith("8-K") and main_doc is None:
            main_doc = href
    docs = exhibits or ([main_doc] if main_doc else [])
    return "\n\n".join(html_to_text(sec_get("https://www.sec.gov" + d).text) for d in docs)


def cached_filing_text(filing: dict) -> str:
    """Download a filing's text once and cache it on disk."""
    path = CACHE_DIR / "docs" / f"{filing['accession']}.txt"
    if path.exists():
        return path.read_text()
    text = get_filing_text(filing["index_url"])
    path.write_text(text)
    return text


def is_relevant(filing: dict) -> bool:
    """Keep original 8-Ks filed recently under items that can carry product news."""
    return (filing["form"] == "8-K"
            and filing["filing_date"] >= MIN_FILING_DATE
            and any(item in filing["items"] for item in RELEVANT_ITEMS))

In [ ]:
candidates = []   # product-related documents that will be sent to the LLM
stats = {"companies": 0, "filings_listed": 0, "filings_downloaded": 0}

for _, company in companies.iterrows():
    if len(candidates) >= MAX_CANDIDATE_DOCS:
        break
    stats["companies"] += 1
    try:
        filings = list_8k_filings(company["cik"])
    except requests.RequestException as err:
        print(f"skip {company['ticker']}: {err}")
        continue
    stats["filings_listed"] += len(filings)
    for filing in filter(is_relevant, filings):
        try:
            text = cached_filing_text(filing)
        except requests.RequestException as err:
            print(f"skip {filing['accession']}: {err}")
            continue
        stats["filings_downloaded"] += 1
        if PRODUCT_KEYWORDS.search(text):
            candidates.append({**filing, "ticker": company["ticker"], "sec_name": company["name"],
                               "text": text[:MAX_DOC_CHARS]})
    if stats["companies"] % 10 == 0:
        print(f"{stats['companies']} companies scanned, {len(candidates)} candidate documents")

stats["candidate_docs"] = len(candidates)
stats["companies_with_candidates"] = len({c["ticker"] for c in candidates})
print(stats)

## 3. LLM extraction with Named Entity Recognition

Each candidate document goes to **Claude Sonnet 5.5** (`claude-sonnet-5-5`) through the Anthropic Python SDK.

* **NER:** the model first lists every **ORGANIZATION** and **PRODUCT** entity in the text
  (e.g. *"Apple Inc."* → ORGANIZATION, *"MacBook Pro"* → PRODUCT).
* **Extraction:** from the PRODUCT entities it keeps only products that are **newly announced, launched or
  approved** in this filing, each with a description under 180 characters.
* **Structured outputs:** the response is constrained to a Pydantic JSON schema (`client.beta.messages.parse`),
  so every answer is valid, parseable JSON with exactly the fields we need.
* **Settings:** adaptive thinking with `effort="low"` (simple extraction task), `fallbacks="default"`
  (server-side retry on another model if a request is declined), 6 parallel requests, answers cached per filing.

In [ ]:
class NewProduct(BaseModel):
    """One newly announced product."""
    new_product: str = Field(description="Name of the newly announced product (PRODUCT entity)")
    product_description: str = Field(description="Concise factual description, under 180 characters")


class FilingExtraction(BaseModel):
    """NER entities and new products found in one 8-K document."""
    company_name: str = Field(description="The filing company (ORGANIZATION entity)")
    organizations: list[str] = Field(description="All ORGANIZATION entities mentioned")
    product_mentions: list[str] = Field(description="All PRODUCT entities mentioned")
    new_products: list[NewProduct] = Field(description="Only products newly announced in this filing")


SYSTEM_PROMPT = """You analyze SEC Form 8-K filings (usually press releases) for a financial dataset.

Step 1 - Named Entity Recognition: list the ORGANIZATION entities (companies, agencies) and the PRODUCT
entities (named products, services, platforms, devices, drugs, software, vehicles) mentioned in the text.

Step 2 - New products: from the PRODUCT entities, keep only products that the filing company announces as
NEW in this document: launched, introduced, unveiled, released, made available or approved by a regulator
(e.g. FDA approval of a drug). Do NOT include existing products that are merely mentioned, financial results,
acquisitions, partnerships without a new product, executive changes, or products of other companies.
If there is no new product, return an empty new_products list.

For each new product give its specific name and a concise factual description under 180 characters
(what it is and what is new). company_name is the filing company's name as written in the document."""

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY


def extract_products(doc: dict) -> dict:
    """Run NER + new-product extraction on one document with the LLM (cached per accession number)."""
    path = CACHE_DIR / "llm" / f"{doc['accession']}.json"
    if path.exists():
        return json.loads(path.read_text())
    response = client.beta.messages.parse(
        model=LLM_MODEL,
        max_tokens=8000,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content":
                   f"Company ticker: {doc['ticker']}\nFiling date: {doc['filing_date']}\n\n"
                   f"<filing>\n{doc['text']}\n</filing>"}],
        output_format=FilingExtraction,
        output_config={"effort": LLM_EFFORT},
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",
    )
    if response.stop_reason == "refusal" or response.parsed_output is None:
        result = {"error": f"no output (stop_reason={response.stop_reason})"}
    else:
        result = response.parsed_output.model_dump()
    result["usage"] = {"input_tokens": response.usage.input_tokens,
                       "output_tokens": response.usage.output_tokens, "model": response.model}
    path.write_text(json.dumps(result))
    return result

In [ ]:
# Quick test on the example from the assignment
example = {"accession": "example-test", "ticker": "AAPL", "filing_date": "2025-02-19",
           "text": "Apple Inc. has announced the launch of its latest MacBook Pro, featuring a new M3 processor."}
print(json.dumps(extract_products(example), indent=2))

In [ ]:
# Run the LLM over all candidate documents in parallel
results = {}
start = time.time()
with ThreadPoolExecutor(max_workers=LLM_WORKERS) as pool:
    futures = {pool.submit(extract_products, doc): doc["accession"] for doc in candidates}
    for i, future in enumerate(as_completed(futures), 1):
        try:
            results[futures[future]] = future.result()
        except anthropic.APIError as err:
            results[futures[future]] = {"error": str(err)}
        if i % 25 == 0:
            print(f"{i}/{len(candidates)} documents processed")

errors = sum("error" in r for r in results.values())
in_tok = sum(r.get("usage", {}).get("input_tokens", 0) for r in results.values())
out_tok = sum(r.get("usage", {}).get("output_tokens", 0) for r in results.values())
print(f"Done in {time.time() - start:.0f}s | documents: {len(results)} | errors: {errors} | "
      f"input tokens: {in_tok:,} | output tokens: {out_tok:,}")

## 4. Clean, structure and save the output

* `company_name` comes from the LLM's NER (ORGANIZATION entity for the filer). If it's blank we use the SEC name.
* `stock_name` is the ticker and `filing_time` is the EDGAR filing date (YYYY-MM-DD). Both come from SEC metadata.
* Descriptions are checked against the 180-character limit and trimmed at a word boundary if needed.
* Duplicates (the same company announcing the same product in several filings) keep the earliest filing.
* Output: `output/new_products.csv` (all values quoted) and `output/new_products_table.txt` (`|`-separated).

In [ ]:
def limit_description(text: str, limit: int = MAX_DESC_LEN) -> str:
    """Ensure the description is shorter than `limit` characters (cut at a word boundary)."""
    text = re.sub(r"\s+", " ", text).strip()
    if len(text) < limit:
        return text
    return text[: limit - 4].rsplit(" ", 1)[0].rstrip(",;:") + "..."


rows = []
for doc in candidates:
    result = results.get(doc["accession"], {})
    for product in result.get("new_products", []):
        if not product["new_product"].strip():
            continue
        rows.append({
            "company_name": result.get("company_name", "").strip() or doc["sec_name"],
            "stock_name": doc["ticker"],
            "filing_time": doc["filing_date"],
            "new_product": product["new_product"].strip(),
            "product_description": limit_description(product["product_description"]),
        })

df = pd.DataFrame(rows, columns=["company_name", "stock_name", "filing_time", "new_product", "product_description"])
df["_key"] = df["stock_name"] + "|" + df["new_product"].str.lower().str.replace(r"[^a-z0-9]", "", regex=True)
df = df.sort_values("filing_time").drop_duplicates("_key").drop(columns="_key")
df = df.sort_values(["filing_time", "stock_name"], ascending=[False, True]).reset_index(drop=True)

assert (df["product_description"].str.len() < MAX_DESC_LEN).all()
print(f"New product entries: {len(df)} from {df['stock_name'].nunique()} companies")

In [ ]:
# CSV with all values quoted, as in the assignment's output format
df.to_csv(OUTPUT_DIR / "new_products.csv", index=False, quoting=csv.QUOTE_ALL)

# Pipe-separated table version
widths = {col: max(len(col), df[col].str.len().max()) for col in df.columns}
header = " | ".join(col.ljust(widths[col]) for col in df.columns)
lines = [header, "-" * len(header)] + [
    " | ".join(str(row[col]).ljust(widths[col]) for col in df.columns) for _, row in df.iterrows()]
(OUTPUT_DIR / "new_products_table.txt").write_text("\n".join(lines))

# Raw NER/LLM output per document, kept for transparency
with open(OUTPUT_DIR / "llm_extractions.jsonl", "w") as f:
    for doc in candidates:
        f.write(json.dumps({"accession": doc["accession"], "ticker": doc["ticker"],
                            "filing_date": doc["filing_date"], **results.get(doc["accession"], {})}) + "\n")

print("\n".join(lines[:22]))

## 5. Summary statistics (for the report)

In [ ]:
docs_with_products = sum(bool(r.get("new_products")) for r in results.values())
ner_match = sum(
    doc["sec_name"].split()[0].lower() in results.get(doc["accession"], {}).get("company_name", "").lower()
    for doc in candidates)

summary = {
    "LLM model": LLM_MODEL,
    "Companies scanned (S&P 500)": stats["companies"],
    "8-K filings listed": stats["filings_listed"],
    "8-K documents downloaded (relevant items)": stats["filings_downloaded"],
    "Documents sent to LLM (keyword pre-filter)": stats["candidate_docs"],
    "Documents with at least one new product": docs_with_products,
    "Companies with new products": df["stock_name"].nunique(),
    "Final CSV rows": len(df),
    "LLM errors / refusals": errors,
    "NER company name agrees with SEC name": f"{ner_match}/{len(candidates)}",
    "Input tokens": in_tok,
    "Output tokens": out_tok,
}
pd.Series(summary, name="value").to_frame()